# A Socio-Economic Model of Segregation, Neighborhood Change and Housing Inequality
## Part 1: Agent-based Model and Computational Experiments
### Malte Grönemann

I formalize the theoretical argument presented in the paper using an agent-based model, where agents interact on a regular spatial grid, as popularized by Schelling (1971). The model is primarily based on the model by Benard and Willer (2007). There are two classes of agents that interact with each other and themselves. Households occupy housing units and seek to live in those that maximize their residential satisfaction, i.e., utility, which is a function of housing quality and neighborhood status, given their budget. They are mobile, so they can move in and out of housing units and neighborhoods when there are vacant units available. Landlords own the housing units, which are represented by the cells on the city grid. They need to decide whether it is profitable to invest in their housing units, which they do heuristically based on average rents in the neighborhood. Housing quality has inertia, though, and is heavily influenced by housing quality at t-1 (path dependency).

The following sections describe the model and the computational experiments. It also provides a single-run animation to foster understanding and offers the opportunity to experiment with the model's parameters. The computational experiments are conducted using separate files without explanations and animations.

In [1]:
import numpy as np
import pandas as pd
import random

from plotly.subplots import make_subplots
import plotly.graph_objects as go

## Defining Agents

### Households

Households are the demanders of housing. Each household $i$ is characterised by its disposable income on housing $w_i \in [0, 1]$ and its social status $s_i \in [0, 1]$. Both household attributes are fixed over time. Income distributions are almost always positively skewed and have long tails. I sample the household's income from a $Beta(2, \ 5)$ distribution to capture this typical feature while cutting down on model sophistication. Income inequality in samples from this distribution, as measured by the Gini index, averages about 31.5, which is comparable to the level of income inequality in Germany, France, and Canada in recent years. I assume a similar distributional form for social status $s_i$, which is typically correlated with income. The social status is therefore sampled dependent on income and a correlation parameter $r \in [0, 1]$:

$$ s_i \sim r \ w_i \\ + \\ (1 - r) \ Beta(2, \ 5) $$

Housing in this model represents a consumption bundle $x$ of two commodities, housing quality $q(x, t)$ and neighborhood status $\bar{s}(x, t)$, which are variable over time $t$. Housing quality is variable because landlords can invest (see section on supply), and neighborhood status is variable over time as households can move into and out of units and neighborhoods. Housing quality represents the desirability of the unit, while neighborhood status represents the desirability of the neighborhood. I operationalize neighborhood status as the average social status of the inhabitants $j$ of the housing units in the Moore neighborhood of the housing unit $x$ that household $i$ considers moving to at time $t$. $n_{x, t}$ is the number of neighbors the household would have at that location, i.e., the number of occupied housing units neighboring $x$ at time $t$. The social position of a neighbor $j$ at location $x$ at time $t$ is referred to by $s_{j}(x, t)$.

$$ \bar{s}(x, t) = \frac{1}{n_{j}(x, t)} \sum_{j = 1}^{n_{j}(x, t)} s_{j} $$

I propose the following Cobb-Douglas utility function to model the household's housing preferences. This common functional form codifies the decreasing marginal utility a household gets from a unit of neighborhood status and housing quality. The parameter $a \in [0, 1]$ weights the two commodities and indicates the relative importance of neighborhoods to housing quality. If $a = 0$, only housing quality matters in the utility calculation of the household; at $a = 1$, only average social status matters. $a$ is identical for all households in a given simulation run; therefore, all housing units are equally desirable to all households.

$$ U(x, t) = \\ \bar{s}(x, t)^a \\ q(x, t)^{1-a} $$

The set of all housing units $x$ at time $t$ is denoted by $X(t)$, and the set of renters inhabiting housing unit $x$ at time $t$ is denoted $L(x, t)$, which can contain at most one renter: $|L(x, t)| \leq 1 \ \forall \ x, t$. For each household $i$, there is a set of available options, the *budget set* $B_i(t) \subseteq X(t)$. The housing units in this set are the household's current and unoccupied locations if the rent $p(x, t)$ of these locations is below the household's disposable income. If the current location becomes too expensive for the household, it is no longer within the budget set, and the household is forced to move (displacement).

$$ B_i(t) = \{x \in X(t): (\ L(x, t) = \{\emptyset\} \ \lor \ L(x, t) = \{i\}) \ \land \ p(x, t) \leq w_i \} $$

Suppose households choose their housing unit according to the discussed preferences. In that case, their choice of housing unit $x$ can be treated *as if* they maximize their utility under the constraint that they need to be able to afford this location:

$$ \operatorname*{arg\,max}_{x \in B_i(t)} U(x, t) \\* $$

If no housing units are available with rent below or equal to the household's disposable income, the household moves to the location with the lowest rent that is available.

In [2]:
class Household:
    """Households are agents that can move around the city. They want to find a residence where at least
    a given proportion (threshold) of neighbors belong to their own race."""
    def __init__(self, model, hh_id, x, y):
        self.model = model
        self.hh_id = hh_id
        self.pos = (x, y)

        r_correlation = self.model.p['r_correlation']
        distribution = self.model.p['distribution']
        # fixed attributes
        self.income = np.random.beta(a = distribution, b = 2.5 * distribution, size = 1)[0]
        self.status = (1 - r_correlation) * np.random.beta(a = distribution, b = 2.5 * distribution, size = 1)[0] + r_correlation * self.income


    def move(self):
        """ Households move to the housing unit that maximises their residential satisfaction given their income.
        If households cannot afford any available housing units, they move to the unit with cheapest rent. """
        choice_set = set([ll for ll in self.model.landlords if ll.empty])
        choice_set.add(self.model.landlord_by_pos[self.pos])
        # Filter choice set by affordability (budget set)
        budget_set = [ll for ll in choice_set if ll.rent <= self.income]
        if budget_set:  # If there are options within the budget
            # Find the housing unit with the maximum utility within the budget
            max_utility_landlord = max(budget_set, key=lambda landlord: landlord.utility)
            housing_unit = max_utility_landlord.pos
        else:  # No affordable options, select the cheapest rent
            min_rent_landlord = min(choice_set, key=lambda landlord: landlord.rent)
            housing_unit = min_rent_landlord.pos
        # Move to the selected housing unit
        old_pos = self.pos
        new_pos = housing_unit
        self.pos = new_pos
        self.model.update_household_position(self, old_pos, new_pos)
        return old_pos, new_pos  # Return both positions for tracking affected landlords in run


### Landlords

Households are the demanders of housing. Each household $i$ is characterised by its disposable income on housing $w_i \in [0, 1]$ and its social status $s_i \in [0, 1]$. Both household attributes are fixed over time. Income distributions are almost always positively skewed and have long tails. I sample the household's income from a $Beta(2, \ 5)$ distribution to capture this typical feature while cutting down on model sophistication. Income inequality in samples from this distribution, as measured by the Gini index, averages about 31.5, which is comparable to the level of income inequality in Germany, France, and Canada in recent years. I assume a similar distributional form for social status $s_i$, which is typically correlated with income. The social status is therefore sampled dependent on income and a correlation parameter $r \in [0, 1]$:

$$ s_i \sim r \ w_i \\ + \\ (1 - r) \ Beta(2, \ 5) $$

Landlords supply housing quality $q(x, t)$ of housing unit $x$. $q(x, t)$ represents the monetary value of housing quality independent of the neighborhood. Quality is path-dependent; it is dependent on past investments. Confronted with fundamental uncertainty about the development of their neighborhood and therefore unknown returns on investments, landlords are sideways-looking and form expectations about neighborhood development based on changes in local rents. Specifically, landlords adjust their housing quality in response to the average rent in the Moore neighborhood. As landlords would like to know whether investing will be profitable, an increase in average neighborhood rent signals to them that their neighborhood attracts wealthy residents who can provide a return on investment. However, if the rents in the neighborhood decrease, returns in this neighborhood are too low to justify investments, and the neighborhood's quality decreases due to material decay. Landlords are responsive to the size of the change in neighborhood rents, however. If neighborhood rents only rise slightly, they will not invest large sums of money. If neighborhood rents decrease slightly, they do some upkeep, but not enough to maintain the previous level of housing quality.

If $x_j$ is one of the eight housing units in the Moore neighborhood of housing unit $x_i$ at time $t$, the average rent $\bar{p}(x_i, t)$ in the neighborhood of $x_i$ at time $t$ is given by:

$$ \bar{p}(x_i, t) = \frac{1}{8} \sum_{k = 1}^{8} p(x_j, t) $$

Given these assumptions, I operationalized the landlords' decision-making process as a weighted average of the previous quality of the housing unit and the average rent in the neighborhood. The weighting parameter $b \in (0, 1)$ represents the inertia of housing quality. If $b$ is 1, housing quality cannot change over time; only when $b$ is below 1 does the development of local rents affect housing quality. The housing quality is then given by:

$$
q(x_i, t) = b \ q(x_i, t-1) \\ + \\ (1 - b) \ \bar{p}(x_i, t)
$$

In [3]:
class Landlord:
    """ Landlord agents are initiated with initially random housing quality.
        Utility and rents are equal to the housing quality at setup.
        Various variables are initiated that are filled at model setup or updated throughout the simulation."""
    def __init__(self, model, x, y):
        self.model = model
        self.x = x
        self.y = y
        self.pos = (x, y)
        self.nb_pos = []
        self.nb_ll = []
        self.empty = True

        distribution = self.model.p['distribution']
        self.housing_quality = np.random.beta(a = distribution, b = 2.5 * distribution, size = 1)[0]
        self.utility = self.housing_quality
        self.rent = self.housing_quality
        self.hh_id = None # updated throughout simulation based on household occupying the unit
        self.hh_income = None
        self.hh_status = None


    def neighborhood(self):
        """ Returns the coordinates of the neighborhood of a given position on a torus. The size of their neighborhood is
        determined by the global parameter vision. The neighborhood is then the Moore neighborhood of all units within a distance of vision."""
        vision = self.model.p['vision']
        size = self.model.p['size']
        return [((self.x + dx) % size, (self.y + dy) % size)
                for dx in range(-vision, vision + 1)
                for dy in range(-vision, vision + 1)
                if not (dx == 0 and dy == 0)]


    def invest(self):
        """ If neighborhood average rent increases, landlords invest in their housing quality proportionally to the change. If average rents decrease, housining quality decreases as well. Housing quality is path dependent, a proportion of the previous quality is retained. Utility is calculated based on the housing quality and the status of the households in the neighborhood."""
        # update quality
        a_preferences = self.model.p['a_preferences']
        b_inertia = self.model.p['b_inertia']
        mean_rent = np.mean([ll.rent for ll in self.nb_ll])
        self.housing_quality = b_inertia * self.housing_quality + (1 - b_inertia) * mean_rent

        # update utility
        hh_status_nb = [ll.hh_status for ll in self.nb_ll]
        hh_status_nb = [x for x in hh_status_nb if not np.isnan(x)]
        if hh_status_nb:
            mean_status = np.mean(hh_status_nb)
        else:
            mean_status = 0
        self.utility = (mean_status ** a_preferences) * (self.housing_quality ** (1 - a_preferences))


    def update_hhvars(self):
        """ I only export data from the landlords. To also have access to the household data, I get the household id, income, and status from the household agent that occupies the landlord's unit."""
        my_renter = self.model.household_by_pos.get(self.pos, None)
        if my_renter:
            self.empty = False
            self.hh_id = my_renter.hh_id
            self.hh_income = my_renter.income
            self.hh_status = my_renter.status
        else:
            self.empty = True
            self.hh_id = np.nan
            self.hh_income = np.nan
            self.hh_status = np.nan


    def update_rent(self): # for substantive description, see next section.
        """ Rents are calculated based on the city-wide distribution of utility and income."""
        competition = self.model.utility_income_df[
            (self.model.utility_income_df['utility'] <= self.utility) &
            self.model.utility_income_df['hh_income'].notna()
            ]['hh_income']
        if competition.empty:
            self.rent = self.model.utility_income_df['hh_income'].min()
        else:
            self.rent = np.percentile(competition, 75)

## Rent

The price or rent the household pays the landlord to live at a given housing unit is a result of supply and demand in competitive markets. Specifically, rent is a mapping of the desirability of the housing unit onto the income distribution. Housing units providing the highest residential satisfaction will face the fiercest competition among interested renters, and those with the highest disposable income can outbid the others. Therefore, the units with the highest residential satisfaction will be the most expensive, while the least expensive have the lowest level of residential satisfaction. To implement such a price-building mechanism, I decided to model rent as the 75th percentile of the incomes of all renters living in a housing unit with a lower residential satisfaction than their unit provides. Renters living in housing units with lower residential satisfaction want to move there because they can increase their residential satisfaction by moving to this housing unit. The renters in better housing units have no incentive to move to this unit. The 75th percentile is somewhat arbitrary, but it gives some leeway for movements to occur, whereas asking the maximum income of renters in a lower utility housing unit as rent makes it very unlikely that a household will move there. The lower rent than potentially achievable reduces the time the landlord needs to wait until a renter willing to pay the asking price moves in.

$C(x_i, t)$ denotes all housing units $x_j$ that have a lower utility than $x_i$, which are the competition of $x_i$. Rent is then the 75th percentile of the incomes of the households $L(C_i)$ living in the housing units in the competition set.

$$ C(x_i, t) = \{ x \in X: U(x_j, t) \leq U(x_i, t)  \} $$

$$ p(x, t) = P_{75}[\{ L(C_i): w_i  \}] $$

In [4]:
## Create a dataframe of all incomes and utilities for rent calculations of households
def utility_income_data(model):
    """ Create a dataframe of all incomes and utilities for rent calculations of landlords. """
    utility_income_df = pd.DataFrame(
        [(ll.utility, ll.hh_income) for ll in model.landlords],
        columns=['utility', 'hh_income']
    )
    return utility_income_df

## Model Setup and Scheduling

The two classes of agents interact on a spatial grid that represents a city. The *size* parameter sets the edge length of the square grid. Each cell of the grid represents one housing unit into which a household can move and a landlord can invest. The number of housing units (and consequently landlords) is *size* squared. To avoid edge effects, the edges of the grid loop around to create a torus. The *density* $\in ]0, 1[$ parameter sets the population density by setting the number of households to *density* * *size* ** 2. Consequently, a proportion of 1 - *density* of the housing units is vacant at every point in time. At setup, exactly one landlord class agent is placed in every grid cell, while the household class agents are distributed randomly in the still empty cells. At every point in time, only one household class agent can occupy a single cell.

At every time step after the setup, all landlords first update their respective variables and decide whether to invest or not. Then, all households decide whether and where to move. After moving (or staying), the households update their respective variables.

If the parameter *turnover* ($\in [0, 0.1]$) is greater than 0, a corresponding proportion of all households is removed from the grid, and newly created households are added to empty housing units. This represents population dynamics, where people move in and out of the city. This creates random moves, which introduce noise. Noise has been shown to affect results of computational models, often improving their empirical fit (Macy and Tsvetkova 2015, Mäs and Helbing 2020). It also ensures that agents move to their optimal location and do not get stuck in suboptimal equilibria. The latter can be achieved with high vacancy rates or population turnover (Fossett and Waren 2005).

There is no natural end to this simulation. After a burn-in period, spatial patterns emerge. The time the model is run and the time disregarded for analysis, as the burn-in, are set arbitrarily in the computational experiments based on visual inspection.

In [5]:
class SocEconHousing:
    """This model is a discrete-time stochastic agent-based model that simulates the socio-economic segregation of a city and the formation of stable distinct neighborhoods, both demographically and in housing. The model is initialized with a given size, density, ... . The model runs for a given number of time steps.
    The parameters are supplied to the model as a dictionary.
    The model's data is exported to a pandas dataframe."""
    def __init__(self, params, sample_id=1):
        self.sample_id = sample_id
        self.p = params
        self.time = 0

        # Initialize agents and dataframe
        self.landlords = []
        self.n_households = int((self.p['size'] ** 2) * self.p['density'])
        self.households = []
        self.df = pd.DataFrame()

        for i in range(self.p['size']):
            for j in range(self.p['size']):
                self.landlords.append(Landlord(self, x=i, y=j))

        for ll in random.sample(self.landlords, self.n_households):
            # hh_id works for sizes below 1000, adapt for larger models to ensure unique hh_ids
            self.households.append(Household(self, hh_id=ll.x*1000+ll.y, x=ll.x, y=ll.y))

        # Add spatial indices
        self.landlord_by_pos = {ll.pos: ll for ll in self.landlords}
        self.household_by_pos = {}
        for hh in self.households:
            self.household_by_pos[hh.pos] = hh

        # Prepare neighborhood variables of the landlords
        for landlord in self.landlords:
            landlord.nb_pos = landlord.neighborhood()
            landlord.nb_ll = [ll for ll in self.landlords if ll.pos in landlord.nb_pos]
            landlord.update_hhvars()


    def update_household_position(self, household, old_pos, new_pos):
        """Update the spatial index when a household moves"""
        if old_pos in self.household_by_pos:
            del self.household_by_pos[old_pos]
        self.household_by_pos[new_pos] = household


    def report(self):
        """ Returns a dataframe with the model's data. """
        current_data = []
        for ll in self.landlords:
            current_data.append([ll.x, ll.y, ll.housing_quality, ll.utility, ll.rent, ll.hh_id, ll.hh_income, ll.hh_status])
        current_df = pd.DataFrame(current_data, columns=['x', 'y', 'housing_quality', 'utility', 'rent', 'hh_id', 'hh_income', 'hh_status'])
        current_df['sample_id'] = self.sample_id
        current_df['time'] = self.time
        current_df = pd.concat([self.df, current_df], ignore_index=True)
        return current_df.reset_index(drop=True)


    def population_dynamics(self):
        """Remove and add households to simulate population dynamics"""
        pop_change = int(self.p['turnover'] * self.n_households)
        # Removing households
        outmovers = random.sample(self.households, pop_change)
        affected_by_removal = set()
        for hh in outmovers:
            if hh.pos in self.household_by_pos:
                old_landlord = self.landlord_by_pos[hh.pos]
                affected_by_removal.add(old_landlord)
                affected_by_removal.update(old_landlord.nb_ll)
                del self.household_by_pos[hh.pos] # Remove from spatial index
        self.households = [hh for hh in self.households if hh not in outmovers] # Remove households from list
        del outmovers # delete agent objects from memory
        for ll in affected_by_removal:
            ll.update_hhvars()
        # Adding households
        available_positions = [ll.pos for ll in self.landlords if ll.empty]
        new_positions = random.sample(available_positions, pop_change)
        affected_by_addition = set()
        for i, pos in enumerate(new_positions):
            x, y = pos
            hh_id = self.time * 10000 + i # Create unique hh_id
            new_hh = Household(self, hh_id, x, y)
            self.households.append(new_hh)
            self.household_by_pos[pos] = new_hh
            new_landlord = self.landlord_by_pos[pos]
            affected_by_addition.add(new_landlord)
            affected_by_addition.update(new_landlord.nb_ll)
        for ll in affected_by_addition:
            ll.update_hhvars()


    def run(self, t_messages=5):
        """ Runs the model for a given number of time steps. It reports the model's status every t_messages time steps.
        Order of operations: Landlords invest first, then households move.
        This creates more realistic, smoother dynamics as landlords make decisions based on
        recent neighborhood trends rather than instantaneous household movements.

        1. Landlords invest (update quality and utility based on previous period)
        2. Landlords update rent (based on new utilities)
        3. Households move (based on updated rents and utilities)
        4. Record data
        5. Population dynamics (if applicable)
        """
        print(f"Model {self.sample_id} started.")
        while self.time < self.p['max_time']:
            if self.time % t_messages == 0:
                print(f"Model {self.sample_id}, step {self.time} out of {self.p['max_time']}")
            # Step 1: Landlords invest based on PREVIOUS period's neighborhood composition
            for ll in self.landlords:
                ll.invest()
            # Step 2: Update rents based on NEW utilities and PREVIOUS period's household distribution
            self.utility_income_df = utility_income_data(self)
            for ll in self.landlords:
                ll.update_rent()
            # Step 3: Households move (based on updated rents and utilities)
            np.random.shuffle(self.households)
            for hh in self.households:
                hh.move()
                for ll in self.landlords:
                    ll.update_hhvars()
            # Step 5: Record data
            self.df = self.report()
            # Step 6: Population dynamics (if turnover > 0)
            if self.p['turnover'] > 0:
                self.population_dynamics()
            # increase time by one
            self.time += 1
        print(f"Model {self.sample_id} finished.")
        return self.df

## Single-run Animation

The following animation shows how spatial patterns in income, status, housing quality, and rents emerge in this model. I chose the parameters for this single animation to reflect the empirically most plausible conditions: households value housing quality more than neighborhood status, and household income and status are highly but not perfectly correlated. Inertia in housing quality is high.

In [6]:
parameters = {
    'r_correlation': 0.5, # correlation between income and status
    'a_preferences': 0.25, # To which extent is the utility determined by neighbors' status
    'distribution': 2, # number between 1 and 5, determines the shape of the Beta dist
    'b_inertia': 0.9, # how much housing quality is retained in one time step
    'density': 0.85, # Density of population
    'size': 30, # Height and length of the grid
    'vision': 1, # distance from the unit the agents consider as neighbors
    'turnover': 0.02, # population dynamics: proportion of households that move in and out of the city
    'max_time': 100  # Maximum number of steps
    }

model = SocEconHousing(params=parameters)
model.run()
results = model.df

Model 1 started.
Model 1, step 0 out of 100
Model 1, step 5 out of 100
Model 1, step 10 out of 100
Model 1, step 15 out of 100
Model 1, step 20 out of 100
Model 1, step 25 out of 100
Model 1, step 30 out of 100
Model 1, step 35 out of 100
Model 1, step 40 out of 100
Model 1, step 45 out of 100
Model 1, step 50 out of 100
Model 1, step 55 out of 100
Model 1, step 60 out of 100
Model 1, step 65 out of 100
Model 1, step 70 out of 100
Model 1, step 75 out of 100
Model 1, step 80 out of 100
Model 1, step 85 out of 100
Model 1, step 90 out of 100
Model 1, step 95 out of 100
Model 1 finished.


In [7]:
grid_size = parameters['size']
time_steps = sorted(results['time'].unique())

variables = ['hh_income', 'hh_status', 'housing_quality', 'rent']
titles = ['Income', 'Status', 'Housing Quality', 'Rent']
all_frames = {var: [] for var in variables}
for t in time_steps:
    time_data = results[results['time'] == t]
    for var in variables:
        grid = time_data.pivot(index='y', columns='x', values=var)
        grid = grid.reindex(index=range(grid_size), columns=range(grid_size), fill_value=np.nan)
        all_frames[var].append(grid.values)
for var in variables:
    all_frames[var] = np.array(all_frames[var])

# Create subplots
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=titles,
    horizontal_spacing=0.05
)

# Add initial heatmaps for each subplot
for i, var in enumerate(variables, 1):
    fig.add_trace(
        go.Heatmap(
            z=all_frames[var][0],
            colorscale='viridis',
            zmin=np.nanmin(results[var]),
            zmax=np.nanmax(results[var]),
            showscale=False,
            hoverinfo='z'
        ),
        row=1, col=i
    )

# Create animation frames
frames = []
for t in range(len(time_steps)):
    frame_data = []
    for i, var in enumerate(variables):
        frame_data.append(
            go.Heatmap(
                z=all_frames[var][t],
                colorscale='viridis',
                zmin=np.nanmin(results[var]),
                zmax=np.nanmax(results[var]),
                showscale=False
            )
        )
    frames.append(go.Frame(data=frame_data, name=str(time_steps[t])))

fig.frames = frames

# Add animation controls
fig.update_layout(
    title_text='Residential Segregation / Spatial Inequalities Over Time',
    updatemenus=[{
        'type': 'buttons',
        'showactive': False,
        'buttons': [
            {
                'label': 'Play',
                'method': 'animate',
                'args': [None, {
                    'frame': {'duration': 100, 'redraw': True},
                    'fromcurrent': True,
                    'mode': 'immediate',
                    'transition': {'duration': 50}
                }]
            },
            {
                'label': 'Pause',
                'method': 'animate',
                'args': [[None], {
                    'frame': {'duration': 0, 'redraw': False},
                    'mode': 'immediate',
                    'transition': {'duration': 0}
                }]
            }
        ],
        'x': 0.1,
        'y': -0.05,
        'xanchor': 'left',
        'yanchor': 'top'
    }],
    sliders=[{
        'active': 0,
        'yanchor': 'top',
        'y': -0.05,
        'xanchor': 'left',
        'currentvalue': {
            'prefix': 'Time Step: ',
            'visible': True,
            'xanchor': 'right'
        },
        'pad': {'b': 10, 't': 50},
        'len': 0.9,
        'x': 0.1,
        'steps': [
            {
                'args': [[f.name], {
                    'frame': {'duration': 0, 'redraw': True},
                    'mode': 'immediate',
                    'transition': {'duration': 0}
                }],
                'label': str(time_steps[k]),
                'method': 'animate'
            }
            for k, f in enumerate(fig.frames)
        ]
    }],
    height=400,
    width=800
)

# Remove axis labels
for i in range(1, 5):
    fig.update_xaxes(showticklabels=False, row=1, col=i)
    fig.update_yaxes(showticklabels=False, row=1, col=i)

fig.write_html('../images/abm_animation.html', auto_play=True) # TODO: change path
fig.show()

## Computational Experiments

The following sections describe the computational experiments I have conducted with this model. The experiments are computationally expensive. I conducted these experiments as separate Python files on the Helix high-performance computing cluster, which is funded by the German federal state of Baden-Württemberg.

### A) Main Experiment

This section describes the setup of the simulation experiment for the main article. The main experiment simulates a 30x30 "city" (grid) with a population density of 85%. Therefore, there are 900 housing units and 765 households in each simulation. Income is distributed according to a Beta(2, 5) distribution, and 2 percent of all households are removed at each time step, with the same number added to the model population dynamics. Households calculate utility, and landlords make their investment decisions based on the Moore neighborhood, considering a distance of one, encompassing the eight neighboring cells. The parameters to vary are the correlation between income and status, as well as the relative importance of housing quality versus the status of neighbors in households' preferences. Both of these parameters vary from 0 to 1 in increments of 0.25, resulting in 5 levels. I also vary the inertia parameter across five levels: 0, 0.8, 0.95, 0.98, and 1. Each parameter combination is run 15 times, resulting in 5 × 5 × 5 × 15 = 1,875 runs. Each run lasts 400 steps, but only the last 100 steps are saved for analysis.

The data are prepared and saved for analysis (see separate files) and reuse by other researchers. As the data record variables from every household and housing unit at every time step, I record 900 housing units x 1875 runs x 400 time steps = 6.75 × 10^8 observations for housing units, of which I analyze 1.6875 × 10^8.

### B) Data Set for Hypothesis Generation

The main experiment and the robustness checks utilize global parameters that span the entire possible range. However, some values of these parameters are implausible to be realised in the real world. I therefore simulate a data set for hypothesis generation that only slightly varies the global parameters around realistic values. As households tend to value housing quality to a greater extent than neighbourhood quality, I use values of 0.2, 0.3, and 0.4 for the relative importance parameter $a$ in the utility function. Income and status are also highly correlated, although not perfectly so. I use correlation parameter values of 0.5, 0.6, and 0.7 (the correlation parameter tends to produce Pearson correlations between the two variables that are higher than the specified parameter value). Housing quality changes slowly over time, so I use inertia parameter values of 0.94, 0.96, and 0.98. I also vary the level of inequality with parameters 1.5, 2, and 2.5.

This selection of parameter values yields 81 combinations, and each combination is executed ten times, resulting in 810 independent simulation runs of 200 steps. With a grid size of 30 x 30, the raw output data amounts to 1.458 × 10^8 observations of housing units, and 7.29 × 10^7 of these are analyzed.

### C) Simulating Gentrification and Decay

Many economic models, including this one, treat gentrification and neighborhood decay not as intrinsic processes of neighborhoods but as responses to changes in overall housing demand in a city. Because of spatial interdependencies, these demand shocks manifest in some selected neighborhoods at a time. To observe which dynamics the model exhibits when confronted with a demand shock, I simulate the empirically most plausible parameter combination for 300 time steps and administer a shock to the overall population density and/or income distribution.

I simulate a 30x30 city with 2% turnover, a vision parameter of 1, a distribution parameter of 2, and a population density of 0.9. The importance of neighbors' social status relative to housing quality is 0.25; status and income have a medium correlation (a parameter value of 0.5, corresponding to a Pearson correlation of 0.71); and housing quality can change slowly (inertia of 0.95). I run the model for longer than usual to ensure there are no trends before the shock. After 300 time steps, the population density can remain stable, decrease to 0.85, or increase to 0.95. Similarly, income inequality can stay at a Gini index of 0.31, increase to 0.42 (distribution = 1), or decrease to 0.26 (distribution = 3). Population density changes exactly at the time of the shock by removing or adding the required number of households. Income inequality gradually changes as the distribution parameter changes for households added to the city through population dynamics. This yields 3×3=9 distinct scenarios, which I repeat 20 times, for a total of 180 runs.

The simulations run for 600 time steps, resulting in a raw data set with 9.72×10^7 observations. I discard the first 200 observations as burn-in, so that I can observe the equilibrium before the shock, the unfolding dynamics, and the equilibrium after the shock. I therefore analyze 6.48x10^7 observations.

### D) Robustness Check: Population Dynamics and Noise

Computational models that converge to an equilibrium may become stuck in a local equilibrium, and some equilibria may not be stable. It has been demonstrated that adding noise to computational experiments yields different, and often better, predictions for real systems (Macy and Tsvetkova 2015, Mäs and Helbing 2020). An intuitive way to think about random events in residential mobility is the movement into and out of the city. The parameter turnover sets the proportion of households that move out of and into the city at every time step, ensuring that the population density remains constant.

The model simulates a grid of 30 x 30 and uses 3 x 3 x 3 = 27 levels of the initial parameters, in addition to 4 levels of turnover: 0, 0.02, 0.05, and 0.1. The model repeats every combination 10 times, so the experiment consists of 18 × 4 × 10 = 720 runs. Visual analysis suggests that the model converges to the equilibrium before 100 steps with this size. Therefore, the model runs 200 steps. The output results in 30 x 30 x 1080 x 200 = 194.4 million observations of housing units, of which 97.2 million are analyzed.

### E) Robustness Check: Income Inequality

It has been established that increases in income inequality lead to increases in income segregation (e.g., Reardon and Bischoff, 2011; Yavas, 2019). This experiment calculates a reduced model that additionally varies the income and status distribution. I set up the Beta distribution in the model to always be unimodal and right-skewed with an expected value of about 0.2857: Beta(a = distribution, b = 2.5 * distribution). The distribution parameter varies only in income inequality, but not in average income.

The model simulates a 30 x 30 grid and utilizes 3 x 3 x 3 = 27 levels of the initial parameters, as well as three levels of inequality. Specifically, I use income distributions of Beta(1, 2.5), Beta(2, 5) and Beta(3, 7.5). These distributions correspond to Gini indices of about 0.42, 0.32, and 0.26, respectively. The model repeats every combination 10 times, so the experiment consists of 27 x 3 x 10 = 810 runs of 200 steps. The output results in 30 x 30 x 540 x 200 = 1.458 × 10^8 observations of housing units, of which I analyze 7.29 × 10^7.

### F) Robustness Check: Vision and Scale of Segregation

In their paper, Laurie and Jaggi (2003) have established that in categorical segregation models, it makes a difference what agents consider their neighborhood. Although their claim that, depending on vision, segregation might not occur at all was shown to be a specificity of their simulation setup (Fossett and Waren, 2005), they additionally find that segregation tends to form larger clusters when vision increases. Before, I have used a Moore neighborhood of distance 1, which refers to the eight bordering (including diagonally) cells. Using the vision parameter, I can increase this distance. I vary it between 1 (8 neighbors), 2 (24 neighbors), and 3 (48 neighbors). This also lends itself to examining the scale of segregation, which has been of interest to scholars (Lee et al. 2008, Reardon et al. 2008, Reardon and Bischoff 2011).

The model simulates a 30x30 grid and uses 3 x 3 x 3 = 27 levels of the initial parameters, in addition to 3 levels of vision. The model repeats every combination 10 times, so the experiment consists of 27 x 3 x 10 = 810 runs of 200 steps. The output results in 30 x 30 x 540 x 200 = 1.458 x 10^8 observations of housing units, of which I analyze 7.29 x 10^7.

### G) Robustness Check: Size and Density

I also want to know whether the size of the grid and the density make a difference in the model. There is also some discussion about whether larger cities tend to be more segregated (e.g., Krupka 2007). The chosen model size for all the other experiments is on the low side, with a 30x30 grid, compared to other segregation ABMs. However, I opted for this lower number because I export observations of all units, not just segregation indices, and I wanted to limit the data size and computation times. The density of 0.85 is standard in segregation ABMs because these models need a relatively high vacancy rate and/or population dynamics to converge to equilibrium (Fossett and Waren 2005). Such high vacancy rates are unrealistic for real cities. The model simulates grids of 10x10, 30x30, and 50x50 and uses 2 x 3 x 2 = 12 levels of the initial parameters, in addition to 3 levels of density (0.85, 0.9, 0.95). The model repeats every combination 10 times, so the experiment consists of 3 × 12 × 3 × 10 = 1080 runs of 200 steps.

### H) Robustness Check: NetLogo Model

To minimize the possibility of coding errors in the ABM and analysis, and to provide researchers with more opportunities to replicate the results, I have also written the ABM in NetLogo. I check whether the models in the two programming languages yield comparable substantial results by running a reduced variant of the main experiment in NetLogo, using a 30x30 grid, a 0.85 population density, a Beta(2, 5) distribution for income and status, and a turnover rate of 2 percent. I vary the correlation and preferences with three levels (0, 0.5, 1 and 0, 0.25, 1, respectively) and the b_inertia parameter with two levels (0.95, 0.8). Each of the nine parameter combinations is repeated 10 times; therefore, I simulate 180 independent runs, each of which runs for 200 steps, of which I discard the first 100 as burn-in. 30 x 30 x 180 runs x 100 analysed steps result in about 16 million observations of housing units.

## References

Benard, Stephen, and Robb Willer. 2007. “A Wealth and Status-Based Model of Residential Segregation.” Journal of Mathematical Sociology 31(2):149–74. doi: 10.1080/00222500601188486.

Fossett, Mark, and Warren Waren. 2005. “Overlooked Implications of Ethnic Preferences for Residential Segregation in Agent-Based Models.” Urban Studies 42(11):1893–1917. doi: 10.1080/00420980500280354.

Krupka, Douglas J. 2007. “Are Big Cities More Segregated? Neighbourhood Scale and the Measurement of Segregation.” Urban Studies 44(1):187–97. doi:10.1080/00420980601023828.

Laurie, Alexander J., and Narendra K. Jaggi. 2003. “Role of ‘Vision’ in Neighbourhood Racial Segregation: A Variant of the Schelling Segregation Model.” Urban Studies 40(13):2687–2704. doi: 10.1080/0042098032000146849.

Lee, Barrett A., Sean F. Reardon, Glenn Firebaugh, Chad R. Farrell, Stephen A. Matthews, and David O’Sullivan. 2008. “Beyond the Census Tract: Patterns and Determinants of Racial Segregation at Multiple Geographic Scales.” American Sociological Review 73(5):766–91. doi:10.1177/000312240807300504.

Macy, Michael, and Milena Tsvetkova. 2015. “The Signal Importance of Noise.” Sociological Methods & Research 44(2):306–28. doi: 10.1177/0049124113508093.

Mäs, Michael, and Dirk Helbing. 2020. “Random Deviations Improve Micro–Macro Predictions: An Empirical Test.” Sociological Methods & Research 49(2):387–417. doi: 10.1177/0049124117729708.

Reardon, Sean F., Stephen A. Matthews, David O’Sullivan, Barrett A. Lee, Glenn Firebaugh, Chad R. Farrell, and Kendra Bischoff. 2008. “The Geographic Scale of Metropolitan Racial Segregation.” Demography 45(3):489–514. doi:10.1353/dem.0.0019.

Reardon, Sean F., and Kendra Bischoff. 2011. “Income Inequality and Income Segregation.” American Journal of Sociology 116(4):1092–1153. doi: 10.1086/657114.

Schelling, Thomas C. 1971. “Dynamic Models of Segregation.” Journal of Mathematical Sociology 1:143–86.

Yavaş, Mustafa. 2019. “Dissecting Income Segregation: Impacts of Concentrated Affluence on Segregation of Poverty.” Journal of Mathematical Sociology 43(1):1–22. doi: 10.1080/0022250X.2018.1476858.